# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [7]:
placement_trainer_agent = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful guide for preparing for job placements.
    Help users with resumes, applications, aptitude tests, interviews,
    communication skills, and professional readiness.
    Give practical, actionable, and concise answers tailored to the user's goals.
    """,
)

In [8]:
student_manager_agent = Agent(
    name="Student Manager",
    instructions="""
   You are a helpful assistant for managing student information and academic progress.
    """,
)

Let's execute the Agent:

In [5]:
with trace("Student Manager Agent"):
    result = await Runner.run(
        student_manager_agent,
        "How can I help you manage student information and academic progress?",
    )

print(result)

RunResult:
- Last agent: Agent(name="Student Manager", ...)
- Final output (str):
    I can help with all aspects of student information and academic progress. Here are some core capabilities:
    
    - Student records: add, update, archive students; manage demographics and contact info.
    - Courses and enrollment: create courses, assign instructors, enroll/withdraw students, track sections.
    - Grades and transcripts: enter grades, compute GPA, calculate weighted/unweighted transcripts, generate progress reports.
    - Attendance and behavior: track attendance, tardies, disciplinary notes.
    - Academic progress: degree audits, progress toward milestones, prerequisite checks.
    - Scheduling and planning: academic plans, course recommendations, holds and alerts.
    - Communications: announcements, reminders, messaging to students or groups.
    - Reporting and exports: grade reports, attendance summaries, CSV exports, analytics dashboards.
    - Compliance and security: role-b

Streaming the answer to the screen, token by token

In [6]:
response_stream = Runner.run_streamed(
    placement_trainer_agent,
    "How can I prepare for a job placement interview?",
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Here’s a concise, practical prep plan you can follow for a job placement interview.

1) Understand the role and company (60 minutes)
- Read the job description; map your experiences to required skills.
- Research: company products, values, competitors, recent news.
- Prepare 2–3 evidence-backed examples showing impact (use STAR: Situation, Task, Action, Result).

2) Prepare your core answers (90–120 minutes total)
- Tell me about yourself: 60-second professional summary aligned to the role.
- Strengths/weaknesses: pick strengths with evidence; select a real weakness and how you’re improving.
- Why us: tie your goals to the company’s mission and the role.
- Why you: unique qualifications and quantified results.
- Behavioral questions: use STAR. Examples you can adapt:
  - “Tell me about a time you faced a challenge at work.”
  - “Describe a time you worked in a team and what you contributed.”
  - “How do you handle tight deadlines?”
- Role-specific: practice 3–5 technical or functional 

_Good Job!_